# Precompute Swing Thresholds

Sweeps a grid of `(pA, pB)` pairs and writes per-set swing quantiles to
`data/swing_thresholds.db`.  The trading bot reads that file via
`trade/swing_thresholds.py` to set its per-game entry threshold dynamically,
rather than using the old static `MIN_GAME_SWING` config.

**Run time:** ~30–60 min at `N_MATCHES = 20_000`.  
Set `N_MATCHES = 5_000` for a quicker first pass (~8–15 min).

**Idempotent:** uses `INSERT OR REPLACE`, so re-running just overwrites existing rows.

In [9]:
import sys, time, sqlite3
from collections import defaultdict

import numpy as np

sys.path.insert(0, r"D:\CMU\Kalshi\Tennis Monte Carlo")
import trade.exact as E

## Parameters

- `PA_GRID` / `PB_GRID`: serve-win probability grids (0.55–0.75, step 0.01)
- `N_MATCHES`: Monte Carlo matches per `(pA, pB, best_of)` cell
- `KEEP_PCTS`: quantile levels stored — one column per level; bot picks the one matching `KEEP_FRACTION` in config
- `DB_PATH`: output SQLite file

In [10]:
PA_GRID   = [round(0.55 + i * 0.01, 2) for i in range(21)]   # 0.55 … 0.75
PB_GRID   = [round(0.55 + i * 0.01, 2) for i in range(21)]
N_MATCHES = 20_000
KEEP_PCTS = [5, 10, 15, 20, 25, 30, 35, 40, 45, 50]
DB_PATH   = r"D:\CMU\Kalshi\Tennis Monte Carlo\data\swing_thresholds.db"

print(f"Grid: {len(PA_GRID)}×{len(PB_GRID)} = {len(PA_GRID)*len(PB_GRID)} pairs")
print(f"Combos (×2 best_of): {len(PA_GRID)*len(PB_GRID)*2}")
print(f"Matches per combo: {N_MATCHES:,}")
print(f"Output: {DB_PATH}")

Grid: 21×21 = 441 pairs
Combos (×2 best_of): 882
Matches per combo: 20,000
Output: D:\CMU\Kalshi\Tennis Monte Carlo\data\swing_thresholds.db


## Simulation Engine

Mirrors the engine in `game_swing_by_set.ipynb`.  Outcomes are drawn from the
exact DP game-win probability, so results are consistent with the model.

In [11]:
class Engine:
    """Memoised DP values at game boundaries for one (pA, pB, best_of) triple."""
    def __init__(self, pA, pB, best_of):
        self.pA, self.pB, self.bo = pA, pB, best_of
        self._v, self._g = {}, {}

    def swing(self, sets, games, p1_serves, tb):
        key = (sets, games, p1_serves, tb)
        if key not in self._v:
            r = E.win_probs(np.array([self.pA]), np.array([self.pB]),
                            sets, games, tb, (0, 0), p1_serves, self.bo)
            self._v[key] = abs(float(r["cond"]["win_game"][0])
                               - float(r["cond"]["lose_game"][0]))
        return self._v[key]

    def hold(self, p1_serves):
        if p1_serves not in self._g:
            self._g[p1_serves] = E.game_win_prob(self.pA if p1_serves else self.pB)
        return self._g[p1_serves]


def set_over(g):
    hi, lo = max(g), min(g)
    return (hi >= 6 and hi - lo >= 2) or hi == 7


def simulate(pA, pB, n_matches, best_of, rng):
    """Return list of (setno, swing) for on-serve, non-tiebreak game boundaries."""
    eng  = Engine(pA, pB, best_of)
    need = best_of // 2 + 1
    recs = []

    for _ in range(n_matches):
        sets      = [0, 0]
        p1_serves = bool(rng.integers(2))
        while max(sets) < need:
            setno = sets[0] + sets[1] + 1
            games = [0, 0]
            while True:
                tb_flag = (games == [6, 6])

                if not tb_flag:
                    sg = games[0] if p1_serves else games[1]
                    rg = games[1] if p1_serves else games[0]
                    on_serve = abs(games[0] - games[1]) <= 1 and sg <= rg
                    if on_serve:
                        recs.append((setno, eng.swing(tuple(sets), tuple(games), p1_serves, False)))

                if tb_flag:
                    p1_win = rng.random() < E.tiebreak_win_prob(pA, pB, a_serves_next=p1_serves)
                    games[0 if p1_win else 1] += 1
                else:
                    holds = rng.random() < eng.hold(p1_serves)
                    if p1_serves:
                        games[0 if holds else 1] += 1
                    else:
                        games[1 if holds else 0] += 1

                p1_serves = not p1_serves
                if set_over(games):
                    break
            sets[0 if games[0] > games[1] else 1] += 1

    return recs

## Database Setup

In [12]:
def init_db(path):
    conn = sqlite3.connect(path)
    conn.execute("""
        CREATE TABLE IF NOT EXISTS thresholds (
            pA       REAL    NOT NULL,
            pB       REAL    NOT NULL,
            best_of  INTEGER NOT NULL,
            set_num  INTEGER NOT NULL,
            keep_pct INTEGER NOT NULL,
            threshold REAL   NOT NULL,
            PRIMARY KEY (pA, pB, best_of, set_num, keep_pct)
        )
    """)
    conn.commit()
    return conn


def write_rows(conn, pA, pB, best_of, recs):
    """Compute quantiles from (setno, swing) records and upsert into the DB."""
    by_set = defaultdict(list)
    for setno, sw in recs:
        by_set[setno].append(sw)

    rows = []
    for setno, swings in sorted(by_set.items()):
        arr = np.array(swings)
        for kp in KEEP_PCTS:
            q = round(float(np.quantile(arr, 1.0 - kp / 100.0)), 3)
            rows.append((round(pA, 2), round(pB, 2), best_of, setno, kp, q))

    conn.executemany(
        "INSERT OR REPLACE INTO thresholds "
        "(pA, pB, best_of, set_num, keep_pct, threshold) VALUES (?,?,?,?,?,?)",
        rows,
    )
    conn.commit()

## Run

Outer loop over all `(pA, pB, best_of)` combos.  Progress is printed every row.

In [13]:
rng   = np.random.default_rng(42)
conn  = init_db(DB_PATH)
total = len(PA_GRID) * len(PB_GRID) * 2
done  = 0
t0    = time.time()

for pA in PA_GRID:
    for pB in PB_GRID:
        for best_of in (3, 5):
            recs = simulate(pA, pB, N_MATCHES, best_of, rng)
            write_rows(conn, pA, pB, best_of, recs)
            done += 1
            elapsed = time.time() - t0
            eta     = elapsed / done * (total - done)
            print(f"[{done:>4}/{total}]  pA={pA:.2f} pB={pB:.2f} Bo{best_of}  "
                  f"elapsed {elapsed:6.0f}s  ETA {eta:6.0f}s", flush=True)

conn.close()
elapsed = time.time() - t0
print(f"\nDone in {elapsed/60:.1f} min.  Written to {DB_PATH}")

[   1/882]  pA=0.55 pB=0.55 Bo3  elapsed      1s  ETA    843s
[   2/882]  pA=0.55 pB=0.55 Bo5  elapsed      3s  ETA   1150s
[   3/882]  pA=0.55 pB=0.56 Bo3  elapsed      3s  ETA   1021s
[   4/882]  pA=0.55 pB=0.56 Bo5  elapsed      5s  ETA   1110s
[   5/882]  pA=0.55 pB=0.57 Bo3  elapsed      6s  ETA   1044s
[   6/882]  pA=0.55 pB=0.57 Bo5  elapsed      8s  ETA   1100s
[   7/882]  pA=0.55 pB=0.58 Bo3  elapsed      8s  ETA   1055s
[   8/882]  pA=0.55 pB=0.58 Bo5  elapsed     10s  ETA   1093s
[   9/882]  pA=0.55 pB=0.59 Bo3  elapsed     11s  ETA   1056s
[  10/882]  pA=0.55 pB=0.59 Bo5  elapsed     12s  ETA   1085s
[  11/882]  pA=0.55 pB=0.60 Bo3  elapsed     13s  ETA   1053s
[  12/882]  pA=0.55 pB=0.60 Bo5  elapsed     15s  ETA   1078s
[  13/882]  pA=0.55 pB=0.61 Bo3  elapsed     16s  ETA   1051s
[  14/882]  pA=0.55 pB=0.61 Bo5  elapsed     17s  ETA   1069s
[  15/882]  pA=0.55 pB=0.62 Bo3  elapsed     18s  ETA   1046s
[  16/882]  pA=0.55 pB=0.62 Bo5  elapsed     20s  ETA   1062s
[  17/88

## Verify

In [14]:
import pandas as pd

conn = sqlite3.connect(DB_PATH)
summary = pd.read_sql_query("""
    SELECT best_of, set_num, keep_pct,
           COUNT(*)       AS n_pairs,
           ROUND(MIN(threshold)*100,1) AS min_pp,
           ROUND(AVG(threshold)*100,1) AS avg_pp,
           ROUND(MAX(threshold)*100,1) AS max_pp
    FROM thresholds
    WHERE keep_pct = 30
    GROUP BY best_of, set_num
    ORDER BY best_of, set_num
""", conn)
conn.close()

print("keep_pct=30 summary (threshold in pp)")
print(summary.to_string(index=False))

keep_pct=30 summary (threshold in pp)
 best_of  set_num  keep_pct  n_pairs  min_pp  avg_pp  max_pp
       3        1        30      441     1.9    15.1    24.0
       3        2        30      441     1.0    14.9    24.0
       3        3        30      441    20.1    37.4    48.2
       5        1        30      441     0.3     9.7    17.9
       5        2        30      441     0.1     9.7    17.9
       5        3        30      441     0.1    11.3    21.9
       5        4        30      441     1.0    14.8    24.0
       5        5        30      441    20.1    37.4    48.2


## Heatmaps — threshold (pp) by pA × pB\n\nOne heatmap per set per format, at `keep_pct = 30`.  Axes are the serve-win\nprobabilities; colour is the entry threshold in percentage points."

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np, sqlite3, pandas as pd

HEATMAP_KEEP_PCT = 30   # which keep_pct level to visualise

conn = sqlite3.connect(DB_PATH)
df   = pd.read_sql_query(
    "SELECT pA, pB, best_of, set_num, threshold FROM thresholds WHERE keep_pct = ?",
    conn, params=(HEATMAP_KEEP_PCT,)
)
conn.close()

pa_vals = sorted(df.pA.unique())
pb_vals = sorted(df.pB.unique())

def make_heatmap(ax, best_of, set_num):
    sub = df[(df.best_of == best_of) & (df.set_num == set_num)]
    pivot = (sub.pivot(index="pB", columns="pA", values="threshold")
               .reindex(index=pb_vals[::-1], columns=pa_vals))   # pB high→low on y
    data = pivot.values * 100   # convert to pp

    vmin = df.threshold.min() * 100
    vmax = df.threshold.max() * 100
    im = ax.imshow(data, aspect="auto", origin="upper",
                   cmap="YlOrRd", vmin=vmin, vmax=vmax,
                   extent=[-0.5, len(pa_vals) - 0.5, -0.5, len(pb_vals) - 0.5])

    # Axis ticks: show every 5th label to avoid crowding
    step = 5
    xticks = range(0, len(pa_vals), step)
    yticks = range(0, len(pb_vals), step)
    ax.set_xticks(list(xticks))
    ax.set_xticklabels([f"{pa_vals[i]:.2f}" for i in xticks], fontsize=7)
    ax.set_yticks(list(yticks))
    ax.set_yticklabels([f"{pb_vals[::-1][i]:.2f}" for i in yticks], fontsize=7)

    ax.set_xlabel("pA (server A)", fontsize=8)
    ax.set_ylabel("pB (server B)", fontsize=8)
    ax.set_title(f"Bo{best_of}  Set {set_num}", fontsize=9, fontweight="bold")
    return im


for best_of, n_sets in ((3, 3), (5, 5)):
    fig, axes = plt.subplots(1, n_sets, figsize=(3.5 * n_sets, 3.8),
                             constrained_layout=True)
    if n_sets == 1:
        axes = [axes]
    im = None
    for set_num, ax in enumerate(axes, start=1):
        im = make_heatmap(ax, best_of, set_num)

    cbar = fig.colorbar(im, ax=axes, shrink=0.85, pad=0.02)
    cbar.set_label("threshold (pp)", fontsize=8)
    cbar.ax.tick_params(labelsize=7)
    fig.suptitle(f"Bo{best_of} entry threshold by serve level  "
                 f"(keep_pct={HEATMAP_KEEP_PCT}%,  floor=12pp)",
                 fontsize=10)
    plt.show()